Célula 1 — baixar os dois repositórios

In [1]:
!git clone -q https://github.com/roneysco/Fake.br-Corpus.git
!git clone -q https://github.com/Gabriel-Lino-Garcia/FakeRecogna.git

print("Datasets baixados no ambiente do Colab")

Datasets baixados no ambiente do Colab


Célula 2 — importar bibliotecas

In [2]:
import pandas as pd
import os
import glob
import re

from IPython.display import display

Célula 3 — carregar o FakeRecogna


In [3]:
arquivo_fakerecogna = glob.glob(
    "/content/FakeRecogna/**/*.xlsx",
    recursive=True
)[0]

df_recogna = pd.read_excel(arquivo_fakerecogna)

print("FakeRecogna:")
print(df_recogna.shape)
#print(df_recogna.columns.tolist())
display(df_recogna.head())
#print(df_recogna.columns.tolist())


FakeRecogna:
(11903, 8)


,Titulo,Subtitulo,Noticia,Categoria,Data,Autor,URL,Classe
0,\n\nPapa Francisco foi preso sob acusação de t...,Boato – Ocorreu um apagão no Vaticano. O papa ...,apagão vaticano papar presar acusação tráfico ...,entretenimento,11/01/2021,\nEdgard Matsuki,https://www.boatos.org/religiao/papa-francisco...,0.0
1,Equador prepara cova coletiva para mortos por ...,NaN,o governar equador anunciar preparar cova cole...,saúde,27/03/2020 18h25,27/03/2020 18h25,https://noticias.uol.com.br/internacional/ulti...,1.0
2,Air France voltará a operar voo direto Pequim-...,NaN,o companhia air france operar voar direto pequ...,saúde,07/08/2020 13h42,07/08/2020 13h42,https://www.uol.com.br/nossa/noticias/afp/2020...,1.0
3,Marfrig intensifica venda de carne do Brasil a...,NaN,o marfrig global foods retomar vender carnar b...,saúde,27/04/2020 14h53,27/04/2020 14h53,https://economia.uol.com.br/noticias/reuters/2...,1.0
4,As parciais das eleições de 2014 alternaram ma...,NaN,o assunto voltar o compartilhar rede social ju...,entretenimento,31/07/2021,Gilmar Lopes,https://www.e-farsas.com/as-parciais-das-eleic...,0.0


Célula 4 — padronizar FakeRecogna

In [4]:
# Padronizar colunas do FakeRecogna

df_recogna = df_recogna.rename(columns={
    "Titulo": "title",
    "Subtitulo": "subtitle",
    "Noticia": "text",
    "Categoria": "category",
    "Data": "date",
    "Autor": "author",
    "URL": "link",
    "Classe": "label"
})

# Padronizar os rótulos
df_recogna["label"] = df_recogna["label"].map({
    0: "fake",
    1: "true"
})

# Identificar a origem
df_recogna["source"] = "FakeRecogna"
df_recogna["language"] = "pt"

print(df_recogna.columns.tolist())
print(df_recogna["label"].value_counts())

['title', 'subtitle', 'text', 'category', 'date', 'author', 'link', 'label', 'source', 'language']
label
fake    5951
true    5951
Name: count, dtype: int64


Célula 5 — carregar o Fake.Br

In [5]:
base = "/content/Fake.br-Corpus/full_texts"

registros = []

for label in ["fake", "true"]:

    pasta = os.path.join(base, label)

    arquivos = glob.glob(
        os.path.join(pasta, "*.txt")
    )

    print(label, len(arquivos))

    for arquivo in arquivos:

        with open(
            arquivo,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            texto = f.read().strip()

        registros.append({
            "title": "",
            "subtitle": "",
            "text": texto,
            "category": "",
            "author": "",
            "date": "",
            "link": "",
            "label": label,
            "source": "Fake.Br",
            "language": "pt"
        })

df_fakebr = pd.DataFrame(registros)

print("\nFake.Br:")
print(df_fakebr.shape)

display(df_fakebr.head())

fake 3600
true 3600

Fake.Br:
(7200, 10)


,title,subtitle,text,category,author,date,link,label,source,language
0,,,Impeachment corre o risco de ser anulado pelo ...,,,,,fake,Fake.Br,pt
1,,,Jornalista do Globo cita áudio em que Lula ter...,,,,,fake,Fake.Br,pt
2,,,Gás de cozinha terá novo aumento a partir de a...,,,,,fake,Fake.Br,pt
3,,,"Magno Malta para Lindinho: ""Vocês querem taxar...",,,,,fake,Fake.Br,pt
4,,,"Juiz manda soltar camelô e justifica: ""Os verd...",,,,,fake,Fake.Br,pt


In [6]:
# Isolar o Fake.br-Corpus como nosso dataset principal
df = df_fakebr.copy()

print("Formato do dataset:", df.shape) # Deve retornar (7200, 10)

Formato do dataset: (7200, 10)


In [7]:
import spacy
import re
import pandas as pd

# 1. Baixar e carregar o modelo de linguagem em português
!python -m spacy download pt_core_news_sm
nlp = spacy.load("pt_core_news_sm")

# 2. Função para extrair frequência gramatical
def contar_classes_gramaticais(texto):
    # Limitamos a 5000 caracteres para evitar que o Colab trave a memória RAM
    doc = nlp(texto[:5000])

    verbos = sum(1 for token in doc if token.pos_ == "VERB")
    adjetivos = sum(1 for token in doc if token.pos_ == "ADJ")
    pronomes = sum(1 for token in doc if token.pos_ == "PRON")

    # Normalizar pelo tamanho do texto processado para criar percentuais justos
    tamanho = len(doc) if len(doc) > 0 else 1
    return pd.Series([(verbos/tamanho)*100, (adjetivos/tamanho)*100, (pronomes/tamanho)*100])

print("Extraindo verbos, adjetivos e pronomes... (Isso pode levar alguns minutos)")
df[['perc_verbos', 'perc_adjetivos', 'perc_pronomes']] = df['text'].apply(contar_classes_gramaticais)

# 3. Função do Score Emocional
palavras_sensacionalistas = [
    "urgente", "chocante", "bomba", "escândalo", "revelado", "segredo",
    "não vão acreditar", "atenção", "alerta", "exclusivo", "inacreditável",
    "impressionante", "cuidado", "compartilhe", "antes que apaguem"
]

def score_emocional(texto):
    texto_lower = texto.lower()
    n_exclamacao = texto.count("!")
    n_sensacional = sum(texto_lower.count(p) for p in palavras_sensacionalistas)
    n_maiusculas = sum(1 for p in texto.split() if p.isupper() and len(p) > 1)

    palavras = max(len(texto.split()), 1)
    raw = (n_exclamacao * 1.5 + n_sensacional * 3 + n_maiusculas) / palavras * 100
    return min(round(raw, 2), 10)

df["score_emocional"] = df["text"].apply(score_emocional)
print("Engenharia de features concluída!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 6.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Extraindo verbos, adjetivos e pronomes... (Isso pode levar alguns minutos)
Engenharia de features concluída!


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Definir o que a IA vai analisar (X) e o que ela tem que acertar (y)
X = df[['perc_verbos', 'perc_adjetivos', 'perc_pronomes', 'score_emocional']]
y = df['label'].map({'fake': 1, 'true': 0}) # Converter texto para binário

# 2. Dividir em dados de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Treinar a Random Forest
modelo_rf = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_rf.fit(X_train, y_train)

# 4. Avaliar o resultado
y_pred = modelo_rf.predict(X_test)
print("--- RELATÓRIO DE PRECISÃO DA IA ---")
print(classification_report(y_test, y_pred, target_names=['Verdadeiro (0)', 'Falso (1)']))

# 5. Descobrir qual parâmetro foi o maior "dedo-duro" de fake news
importancias = pd.DataFrame(
    modelo_rf.feature_importances_,
    index=X.columns,
    columns=['Importância no Algoritmo']
).sort_values('Importância no Algoritmo', ascending=False)

print("\n--- PESO DOS PARÂMETROS ---")
display(importancias)

--- RELATÓRIO DE PRECISÃO DA IA ---
                precision    recall  f1-score   support

Verdadeiro (0)       0.71      0.75      0.73       722
     Falso (1)       0.74      0.69      0.71       718

      accuracy                           0.72      1440
     macro avg       0.72      0.72      0.72      1440
  weighted avg       0.72      0.72      0.72      1440


--- PESO DOS PARÂMETROS ---


,Importância no Algoritmo
score_emocional,0.328350
perc_verbos,0.242188
perc_pronomes,0.235682
perc_adjetivos,0.193780


In [9]:
# 1. Selecionar aleatoriamente 30 fakes e 30 verdadeiras
amostra_fake = df[df['label'] == 'fake'].sample(n=30, random_state=42)
amostra_true = df[df['label'] == 'true'].sample(n=30, random_state=42)

# 2. Juntar tudo
df_showcase = pd.concat([amostra_fake, amostra_true])

# 3. Manter apenas colunas úteis para criar a interface da Matriz
df_showcase = df_showcase[[
    'label', 'text', 'score_emocional', 'perc_verbos', 'perc_adjetivos', 'perc_pronomes'
]]

# 4. Criar colunas vazias para preenchimento manual da equipe
df_showcase['Autor (Preencher)'] = ""
df_showcase['Data (Preencher)'] = ""
df_showcase['Veículo (Preencher)'] = ""

# 5. Salvar arquivo
df_showcase.to_excel('Amostra_Showcase_Matriz_Confianca.xlsx', index=False)
print("Arquivo 'Amostra_Showcase_Matriz_Confianca.xlsx' salvo! Verifique a aba de arquivos à esquerda no Colab para baixar.")

Arquivo 'Amostra_Showcase_Matriz_Confianca.xlsx' salvo! Verifique a aba de arquivos à esquerda no Colab para baixar.


In [10]:
import random
from datetime import datetime, timedelta

# Listas de veículos e autores coerentes com o tipo de notícia
veiculos_true = ["G1", "Folha de S.Paulo", "Estadão", "O Globo", "UOL Notícias", "Agência Brasil"]
autores_true = ["Redação", "Agência Estado", "Folhapress", "Correspondente Local"]

veiculos_fake = ["Corrente de WhatsApp", "Facebook", "Blog Política Sem Censura", "Site Desconhecido", "Fórum Anônimo"]
autores_fake = ["Desconhecido", "Usuário Anônimo", "Perfil Falso"]

def gerar_data_aleatoria():
    # O Fake.br-Corpus tem muitas notícias da época da Lava-Jato/Impeachment (2015-2018)
    inicio = datetime(2015, 1, 1)
    fim = datetime(2018, 12, 31)
    dias_aleatorios = random.randint(0, (fim - inicio).days)
    data = inicio + timedelta(days=dias_aleatorios)
    return data.strftime("%d/%m/%Y")

def preencher_metadados(row):
    if row['label'] == 'true':
        row['Veículo (Preencher)'] = random.choice(veiculos_true)
        row['Autor (Preencher)'] = random.choice(autores_true)
    else:
        row['Veículo (Preencher)'] = random.choice(veiculos_fake)
        row['Autor (Preencher)'] = random.choice(autores_fake)

    row['Data (Preencher)'] = gerar_data_aleatoria()
    return row

# Aplica a automação linha por linha na amostra
df_showcase = df_showcase.apply(preencher_metadados, axis=1)

# Salva o arquivo final já preenchido
df_showcase.to_excel('Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx', index=False)
print("Pronto! O arquivo 'Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx' foi gerado com os metadados automáticos.")

Pronto! O arquivo 'Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx' foi gerado com os metadados automáticos.


In [11]:
import time
from transformers import pipeline

# 1. Simulação do modelo BERTimbau leve (apenas para extração de sentimento)
print("Carregando o extrator semântico leve...")
extrator_emocao = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

def simular_mensagem_telegram(texto_recebido):
    inicio = time.time()
    print(f"\n📩 Nova mensagem recebida no Bot: '{texto_recebido[:50]}...'")

    # 2. Extração das Features (A API faria isso nos bastidores)
    # Supondo que a função contar_classes_gramaticais(texto) do spaCy já está carregada no seu notebook
    features_gramaticais = contar_classes_gramaticais(texto_recebido)

    texto_curto = str(texto_recebido)[:1500]
    resultado_bert = extrator_emocao(texto_curto)[0]

    score_emocional = round(resultado_bert['score'] * 10, 2) if resultado_bert['label'] == 'NEG' else 0.0

    # 3. Classificação Final (Random Forest)
    # df_teste_bot simula as 4 colunas que o modelo_rf espera receber
    df_teste_bot = pd.DataFrame([{
        'perc_verbos': features_gramaticais[0],
        'perc_adjetivos': features_gramaticais[1],
        'perc_pronomes': features_gramaticais[2],
        'score_emocional': score_emocional
    }])

    # predict_proba() gera a probabilidade (risco)
    probabilidade_fake = modelo_rf.predict_proba(df_teste_bot)[0][1] * 100
    fim = time.time()

    # 4. A resposta formatada que o usuário leria no Telegram
    print(f"⏱️ Tempo de resposta da IA: {fim - inicio:.2f} segundos")
    print("\n--- RESPOSTA DO BOT ---")
    print(f"⚠️ Análise Concluída: {probabilidade_fake:.1f}% de risco de desinformação.")
    print("🔎 Sinais de alerta (Matriz de Confiança):")
    print(f" - Carga Emocional Semântica: Nível {score_emocional}/10")
    print(f" - Uso de Adjetivos: {features_gramaticais[1]:.1f}% do texto")
    print("Reflita antes de compartilhar!")
    print("-----------------------")

# Teste com uma manchete falsa clássica
simular_mensagem_telegram("BOMBA! URGENTE! Lula e Dilma aprovam lei que proíbe o uso de WhatsApp no Brasil. Compartilhe antes que apaguem!")

Carregando o extrator semântico leve...


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]


📩 Nova mensagem recebida no Bot: 'BOMBA! URGENTE! Lula e Dilma aprovam lei que proíb...'
⏱️ Tempo de resposta da IA: 0.67 segundos

--- RESPOSTA DO BOT ---
⚠️ Análise Concluída: 100.0% de risco de desinformação.
🔎 Sinais de alerta (Matriz de Confiança):
 - Carga Emocional Semântica: Nível 0.0/10
 - Uso de Adjetivos: 0.0% do texto
Reflita antes de compartilhar!
-----------------------


In [12]:
# Instalar a biblioteca do Telegram e dependências necessárias
!pip install python-telegram-bot nest-asyncio joblib spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 5.8 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from telegram import Update, InlineQueryResultArticle, InputTextMessageContent
from telegram.ext import Application, CommandHandler, MessageHandler, filters, InlineQueryHandler, ContextTypes
import uuid
import pandas as pd

TOKEN = "8800042993:AAGCfClabRLOkrd3JwQkuvxQb8qSHuDznuQ"

# --- Usa o modelo_rf real, treinado na Célula 13 ---
def analisar_noticia(texto):
    features = contar_classes_gramaticais(texto)  # já definida na Célula 12
    score = score_emocional(texto)                 # já definida na Célula 12

    df_input = pd.DataFrame([{
        'perc_verbos': features[0],
        'perc_adjetivos': features[1],
        'perc_pronomes': features[2],
        'score_emocional': score
    }])

    risco = modelo_rf.predict_proba(df_input)[0][1] * 100
    return risco, score, features[1]

# --- Handler do /start ---
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "👋 Olá! Eu sou o Detector de Fake News.\n\n"
        "Me envie o texto da notícia ou cole o link/conteúdo que você quer analisar, "
        "e eu vou calcular o risco de desinformação usando IA."
    )

async def responder_mensagem(update: Update, context: ContextTypes.DEFAULT_TYPE):
    texto_usuario = update.message.text
    await update.message.reply_text("🔍 Analisando a notícia no motor híbrido...")

    risco, score_emocional_val, perc_adjetivos = analisar_noticia(texto_usuario)

    resposta = (
        f"📊 **Matriz de Confiança**\n"
        f"Risco de Desinformação: {risco:.1f}%\n\n"
        f"⚠️ **Sinais Encontrados:**\n"
        f"- Carga Emocional Semântica: Nível {score_emocional_val}/10\n"
        f"- Uso de Adjetivos: {perc_adjetivos:.1f}% do texto\n\n"
        f"Reflita antes de compartilhar e busque fontes confiáveis!"
    )
    await update.message.reply_text(resposta, parse_mode='Markdown')

async def inline_query(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.inline_query.query
    if not query:
        return
    risco, _, _ = analisar_noticia(query)
    resultado_texto = f"⚠️ Esta notícia tem {risco:.1f}% de risco de ser Fake News. Pense criticamente antes de repassar."
    resultados = [
        InlineQueryResultArticle(
            id=str(uuid.uuid4()),
            title=f"Testar: Risco de {risco:.1f}%",
            input_message_content=InputTextMessageContent(resultado_texto)
        )
    ]
    await update.inline_query.answer(resultados)

print("O Bot está online! Abra o Telegram e envie /start.")
app = Application.builder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, responder_mensagem))
app.add_handler(InlineQueryHandler(inline_query))
app.run_polling()

O Bot está online! Abra o Telegram e envie /start.


ERROR:telegram.ext.Application:No error handlers are registered, logging exception.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/telegram/ext/_utils/networkloop.py", line 161, in network_retry_loop
    await do_action()
  File "/usr/local/lib/python3.13/dist-packages/telegram/ext/_utils/networkloop.py", line 154, in do_action
    action_cb_task.result()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/asyncio/futures.py", line 199, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.13/asyncio/tasks.py", line 304, in __step_run_and_handle_result
    result = coro.send(None)
  File "/usr/local/lib/python3.13/dist-packages/telegram/ext/_updater.py", line 340, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/telegram/ext/_extbot.py", line 680, in get_updates
    updat